# PIPELINE DE PREPRODUCCIÓN — Riesgo de Crédito

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, Binarizer, MinMaxScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, mean_absolute_error

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

csv_path = PROJECT_ROOT / "02_datos" / "01_Originales" / "prestamos.csv"
print(f"CSV esperado en: {csv_path.resolve()}")

CSV esperado en: C:\Users\isaac\Google Drive\DS4B\CursoMachineLearningPython2\Casos\ML\03_Riesgos\02_datos\01_Originales\prestamos.csv


In [2]:
# ── A_01 | Importación de datos ──────────────────────────────────────────────
datos = pd.read_csv(csv_path)
val = datos.sample(frac=0.3)
val.to_pickle(PROJECT_ROOT / "02_datos" / "02_Validacion" / "validacion.pkl")
trabajo = datos.loc[~datos.index.isin(val.index)]
print(f"trabajo: {trabajo.shape}, val: {val.shape}")

trabajo: (140000, 25), val: (60000, 25)


In [3]:
# ── A_02 | Calidad de datos ───────────────────────────────────────────────────
df = trabajo.copy()
df.reset_index(drop=True, inplace=True)
df.drop(columns='id_prestamo', inplace=True)
cat = df.select_dtypes(exclude='number').copy()
num = df.select_dtypes(include='number').copy()
cat.drop(columns=['descripcion', 'empleo'], inplace=True)
cat['antigüedad_empleo'] = cat['antigüedad_empleo'].fillna('desconocido')
num.fillna(0, inplace=True)
num_imputar_ceros = ['num_hipotecas', 'num_lineas_credito', 'num_cancelaciones_12meses',
                     'num_derogatorios', 'num_meses_desde_ult_retraso']
num[num_imputar_ceros] = num[num_imputar_ceros].astype(int)
# --- transformaciones de fila ---
a_eliminar = num.loc[num.ingresos > 400000].index.values
cat = cat[~cat.index.isin(a_eliminar)]
num = num[~num.index.isin(a_eliminar)]
print(f"cat: {cat.shape}, num: {num.shape}")

cat: (139599, 7), num: (139599, 15)


In [4]:
# ── A_03 | EDA — Transformaciones ────────────────────────────────────────────
num['DTI'] = np.clip(num['dti'], 0, 100)
num['num_hipotecas'] = np.clip(num['num_hipotecas'], 0, 7)
num['porc_uso_revolving'] = np.clip(num['porc_uso_revolving'], 0, 100)
print(f"cat: {cat.shape}, num: {num.shape}")

cat: (139599, 7), num: (139599, 16)


In [5]:
# ── A_04 | Ingeniería de variables ───────────────────────────────────────────

# Variable objetivo PD
cat['target_pd'] = np.where(
    cat.estado.isin(['Charged Off',
                     'Does not meet the credit policy. Status:Charged Off',
                     'Default']), 1, 0)
cat.drop(columns='estado', inplace=True)

# Variables objetivo EAD y LGD
num['pendiente'] = num.principal - num.imp_amortizado
num['target_ead'] = num.pendiente / num.principal
num['target_lgd'] = 1 - (num.imp_recuperado / num.pendiente)
num['target_lgd'] = num['target_lgd'].fillna(0)
num.target_ead = np.where(num.target_ead < 0, 0, num.target_ead)
num.target_ead = np.where(num.target_ead > 1, 1, num.target_ead)
num.target_lgd = np.where(num.target_lgd < 0, 0, num.target_lgd)
num.target_lgd = np.where(num.target_lgd > 1, 1, num.target_lgd)
num.drop(columns='num_meses_desde_ult_retraso', inplace=True)

# Limpieza de categorías
cat.vivienda = cat.vivienda.replace(['ANY', 'NONE', 'OTHER'], 'MORTGAGE')
cat.finalidad = cat.finalidad.replace(['wedding', 'educational', 'renewable_energy'], 'other')

# OneHotEncoder
var_ohe = ['ingresos_verificados', 'vivienda', 'finalidad', 'num_cuotas']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_ohe = ohe.fit_transform(cat[var_ohe])
cat_ohe = pd.DataFrame(cat_ohe, columns=ohe.get_feature_names_out())

# OrdinalEncoder
var_oe = ['antigüedad_empleo', 'rating']
orden_antigüedad_empleo = ['desconocido', '< 1 year', '1 year', '2 years', '3 years',
                           '4 years', '5 years', '6 years', '7 years', '8 years',
                           '9 years', '10+ years']
orden_rating = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
oe = OrdinalEncoder(categories=[orden_antigüedad_empleo, orden_rating],
                    handle_unknown='use_encoded_value', unknown_value=-1)
cat_oe = oe.fit_transform(cat[var_oe])
nombres_oe = [variable + '_oe' for variable in var_oe]
cat_oe = pd.DataFrame(cat_oe, columns=nombres_oe)

# Binarizer
var_bin = ['num_derogatorios']
bin_ = Binarizer(threshold=0)
num_bin = bin_.fit_transform(num[var_bin])
nombres_bin = [variable + '_bin' for variable in var_bin]
num_bin = pd.DataFrame(num_bin, columns=nombres_bin)

# MinMaxScaler
num_escalar = num[['ingresos',
                   'dti',
                   'num_lineas_credito',
                   'porc_uso_revolving',
                   'principal',
                   'tipo_interes',
                   'imp_cuota']].reset_index(drop=True)
df_res = pd.concat([cat_oe, num_escalar], axis=1)
var_mms = df_res.columns
mms = MinMaxScaler()
df_mms = mms.fit_transform(df_res[var_mms])
nombres_mms = [variable + '_mms' for variable in var_mms]
df_mms = pd.DataFrame(df_mms, columns=nombres_mms)

# Tablón PD
incluir_pd = [num[['id_cliente']].reset_index(drop=True), cat_ohe, df_mms,
              num_bin, cat.reset_index()[['target_pd']]]
df_pd = pd.concat(incluir_pd, axis=1)
df_pd.set_index('id_cliente', inplace=True)

# Tablón EAD (solo registros en default)
incluir_ead = [cat_ohe, df_mms, num_bin,
               num.reset_index()[['id_cliente', 'target_ead']],
               cat.reset_index()[['target_pd']]]
df_ead = pd.concat(incluir_ead, axis=1)
df_ead.set_index('id_cliente', inplace=True)
# --- transformaciones de fila ---
df_ead = df_ead[df_ead.target_pd == 1].drop(columns='target_pd')

# Tablón LGD (solo registros en default)
incluir_lgd = [cat_ohe, df_mms, num_bin,
               num.reset_index()[['id_cliente', 'target_lgd']],
               cat.reset_index()[['target_pd']]]
df_lgd = pd.concat(incluir_lgd, axis=1)
df_lgd.set_index('id_cliente', inplace=True)
# --- transformaciones de fila ---
df_lgd = df_lgd[df_lgd.target_pd == 1].drop(columns='target_pd')

print(f"df_pd: {df_pd.shape}, df_ead: {df_ead.shape}, df_lgd: {df_lgd.shape}")

df_pd: (139599, 30), df_ead: (16629, 30), df_lgd: (16629, 30)


In [6]:
# ── A_05 | Modelo PD — Probabilidad de Default ───────────────────────────────
df = df_pd
x = df.drop(columns='target_pd')
y = df.target_pd
train_x, val_x, train_y, val_y = train_test_split(x, y, test_size=0.3)

pipe = Pipeline([('algoritmo', LogisticRegression())])
grid = [{'algoritmo': [LogisticRegression(solver='saga')],
         'algoritmo__l1_ratio': [0, 0.5, 1],
         'algoritmo__C': [0.01, 0.25, 0.5, 0.75, 1]}]
grid_search = GridSearchCV(estimator=pipe,
                           param_grid=grid,
                           scoring='roc_auc',
                           cv=5,
                           verbose=0,
                           n_jobs=-1)
modelo = grid_search.fit(train_x, train_y)

rl = LogisticRegression(solver='saga', C=1, l1_ratio=1)
rl.fit(train_x, train_y)
pred = rl.predict_proba(val_x)[:, 1]
roc_auc_score(val_y, pred)

0.6953297939489901

In [7]:
# ── A_06 | Modelo EAD — Exposure at Default ──────────────────────────────────
df = df_ead
x = df.drop(columns='target_ead')
y = df.target_ead
train_x, val_x, train_y, val_y = train_test_split(x, y, test_size=0.3)

pipe = Pipeline([('algoritmo', Ridge())])
grid = [{'algoritmo': [Ridge()],
         'algoritmo__alpha': list(np.arange(0.1, 1.1, 0.1))},
        {'algoritmo': [Lasso()],
         'algoritmo__alpha': list(np.arange(0.1, 1.1, 0.1))},
        {'algoritmo': [HistGradientBoostingRegressor(min_samples_leaf=50,
                                                     scoring='neg_mean_absolute_percentage_error')],
         'algoritmo__learning_rate': [0.01, 0.025, 0.05, 0.1],
         'algoritmo__max_iter': [50, 100, 200],
         'algoritmo__max_depth': [5, 10, 20],
         'algoritmo__l2_regularization': [0, 0.25, 0.5, 0.75, 1]}]
grid_search = GridSearchCV(estimator=pipe,
                           param_grid=grid,
                           scoring='neg_mean_absolute_error',
                           cv=3,
                           verbose=0,
                           n_jobs=-1)
modelo = grid_search.fit(train_x, train_y)

modelo_ead = HistGradientBoostingRegressor(learning_rate=0.1,
                                           max_iter=100,
                                           max_depth=5,
                                           min_samples_leaf=50,
                                           scoring='neg_mean_absolute_percentage_error',
                                           l2_regularization=1)
modelo_ead.fit(train_x, train_y)
pred = modelo_ead.predict(val_x)
mean_absolute_error(val_y, pred)

0.15591392782649674

In [8]:
# ── A_07 | Modelo LGD — Loss Given Default ───────────────────────────────────
df = df_lgd
x = df.drop(columns='target_lgd')
y = df.target_lgd
train_x, val_x, train_y, val_y = train_test_split(x, y, test_size=0.3)

pipe = Pipeline([('algoritmo', Ridge())])
grid = [{'algoritmo': [Ridge()],
         'algoritmo__alpha': list(np.arange(0.1, 1.1, 0.1))},
        {'algoritmo': [Lasso()],
         'algoritmo__alpha': list(np.arange(0.1, 1.1, 0.1))},
        {'algoritmo': [HistGradientBoostingRegressor(min_samples_leaf=50,
                                                     scoring='neg_mean_absolute_percentage_error')],
         'algoritmo__learning_rate': [0.01, 0.025, 0.05, 0.1],
         'algoritmo__max_iter': [50, 100, 200],
         'algoritmo__max_depth': [5, 10, 20],
         'algoritmo__l2_regularization': [0, 0.25, 0.5, 0.75, 1]}]
grid_search = GridSearchCV(estimator=pipe,
                           param_grid=grid,
                           scoring='neg_mean_absolute_error',
                           cv=3,
                           verbose=0,
                           n_jobs=-1)
modelo = grid_search.fit(train_x, train_y)

modelo_lgd = HistGradientBoostingRegressor(learning_rate=0.01,
                                           max_iter=100,
                                           max_depth=5,
                                           min_samples_leaf=50,
                                           scoring='neg_mean_absolute_percentage_error',
                                           l2_regularization=0.5)
modelo_lgd.fit(train_x, train_y)
pred = modelo_lgd.predict(val_x)
mean_absolute_error(val_y, pred)

0.08780706280217675